# DETR3D End-to-End Walkthrough

This notebook is meant to be read top to bottom. It starts from raw nuScenes files on disk, constructs a DETR3D-ready PyTorch batch, then defines each architecture component from scratch one by one.

## Notebook Roadmap

1. Raw nuScenes data layout
2. Metadata table loading from scratch
3. Sample linkage across cameras
4. Calibration and projection matrices
5. Ground-truth target construction
6. Image preprocessing and collate
7. Backbone
8. FPN
9. Reference points and feature sampling
10. Cross-attention
11. Decoder layer and transformer
12. Detection head and full model
13. Matcher and loss
14. Training step and experiments

In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset


## 1. Raw nuScenes Data Layout

The extracted dataset root should look like this:

```text
nuscenes/
├── maps/
├── samples/
│   ├── CAM_FRONT/
│   ├── CAM_FRONT_LEFT/
│   ├── CAM_FRONT_RIGHT/
│   ├── CAM_BACK/
│   ├── CAM_BACK_LEFT/
│   └── CAM_BACK_RIGHT/
├── sweeps/
├── v1.0-trainval/
│   ├── sample.json
│   ├── sample_data.json
│   ├── sample_annotation.json
│   ├── calibrated_sensor.json
│   ├── ego_pose.json
│   ├── sensor.json
│   ├── category.json
│   └── ...
└── v1.0-test/
```

The core idea is: images live in `samples/` or `sweeps/`, but the training labels, camera calibration, and ego poses live in the JSON tables under `v1.0-trainval/`.

In [ ]:
DATAROOT = Path('/home/user/datasets/nuscenes')
VERSION = 'v1.0-trainval'
META_ROOT = DATAROOT / VERSION

print(DATAROOT)
print(META_ROOT)
print((DATAROOT / 'samples').exists(), (META_ROOT / 'sample.json').exists())

## 2. Metadata Table Loading From Scratch

Instead of relying on the nuScenes SDK first, we make the structure explicit. The objective is to understand how a `sample` points to camera frames, annotations, calibrations, and poses.

In [ ]:
def load_table(meta_root: Path, name: str):
    path = meta_root / f'{name}.json'
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def build_index(rows: List[dict], key: str = 'token') -> Dict[str, dict]:
    return {row[key]: row for row in rows}


sample_table = load_table(META_ROOT, 'sample')
sample_data_table = load_table(META_ROOT, 'sample_data')
sample_annotation_table = load_table(META_ROOT, 'sample_annotation')
calibrated_sensor_table = load_table(META_ROOT, 'calibrated_sensor')
ego_pose_table = load_table(META_ROOT, 'ego_pose')
sensor_table = load_table(META_ROOT, 'sensor')
category_table = load_table(META_ROOT, 'category')

sample_by_token = build_index(sample_table)
sample_data_by_token = build_index(sample_data_table)
annotation_by_token = build_index(sample_annotation_table)
calibrated_sensor_by_token = build_index(calibrated_sensor_table)
ego_pose_by_token = build_index(ego_pose_table)
sensor_by_token = build_index(sensor_table)
category_by_token = build_index(category_table)

len(sample_table), len(sample_data_table), len(sample_annotation_table)

## 3. Linking One Sample to Its 6 Cameras

A nuScenes `sample` contains a dictionary called `data`. For DETR3D, we care mainly about the six camera tokens:
- `CAM_FRONT`
- `CAM_FRONT_LEFT`
- `CAM_FRONT_RIGHT`
- `CAM_BACK`
- `CAM_BACK_LEFT`
- `CAM_BACK_RIGHT`

In [ ]:
CAMERA_NAMES = [
    'CAM_FRONT',
    'CAM_FRONT_LEFT',
    'CAM_FRONT_RIGHT',
    'CAM_BACK',
    'CAM_BACK_LEFT',
    'CAM_BACK_RIGHT',
]


def get_camera_records(sample_record: dict, sample_data_by_token: Dict[str, dict]):
    camera_records = {}
    for camera_name in CAMERA_NAMES:
        sample_data_token = sample_record['data'][camera_name]
        camera_records[camera_name] = sample_data_by_token[sample_data_token]
    return camera_records


sample_record = sample_table[0]
camera_records = get_camera_records(sample_record, sample_data_by_token)
{name: record['filename'] for name, record in camera_records.items()}

## 4. Calibration and Projection Matrices

DETR3D projects a 3D reference point into each camera image, then samples image features at the projected location. So the data pipeline must build per-camera projection matrices.

We will use `lidar2img` as the conventional key name in `img_metas`, even if we train a camera-only model, because DETR3D codebases often project from a common ego or lidar-aligned 3D frame into each image.

In [ ]:
def quaternion_to_rotation_matrix(q):
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ], dtype=np.float32)


def pose_to_matrix(rotation, translation):
    matrix = np.eye(4, dtype=np.float32)
    matrix[:3, :3] = quaternion_to_rotation_matrix(rotation)
    matrix[:3, 3] = np.asarray(translation, dtype=np.float32)
    return matrix


def invert_se3(matrix):
    rotation = matrix[:3, :3]
    translation = matrix[:3, 3]
    inv = np.eye(4, dtype=np.float32)
    inv[:3, :3] = rotation.T
    inv[:3, 3] = -rotation.T @ translation
    return inv


def build_lidar2img(camera_record):
    calibrated_sensor = calibrated_sensor_by_token[camera_record['calibrated_sensor_token']]
    ego_pose = ego_pose_by_token[camera_record['ego_pose_token']]

    cam_to_ego = pose_to_matrix(calibrated_sensor['rotation'], calibrated_sensor['translation'])
    ego_to_global = pose_to_matrix(ego_pose['rotation'], ego_pose['translation'])

    global_to_ego = invert_se3(ego_to_global)
    ego_to_cam = invert_se3(cam_to_ego)

    intrinsic = np.eye(4, dtype=np.float32)
    intrinsic[:3, :3] = np.asarray(calibrated_sensor['camera_intrinsic'], dtype=np.float32)
    return intrinsic @ ego_to_cam @ global_to_ego


lidar2img = np.stack([build_lidar2img(record) for record in camera_records.values()], axis=0)
lidar2img.shape

## 5. Building 3D Targets From Raw Annotations

A DETR3D training target needs 3D boxes and class ids in one consistent frame. We will use the ego frame and encode each GT box as `[x, y, z, w, l, h, yaw]`.

In [ ]:
NUSCENES_CLASSES = [
    'car', 'truck', 'bus', 'trailer', 'construction_vehicle',
    'pedestrian', 'motorcycle', 'bicycle', 'traffic_cone', 'barrier'
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(NUSCENES_CLASSES)}


def category_to_detection_class(category_name: str):
    for class_name in NUSCENES_CLASSES:
        if category_name.startswith(class_name):
            return class_name
    return None


def yaw_from_quaternion(q):
    w, x, y, z = q
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    return math.atan2(siny_cosp, cosy_cosp)


def build_gt_targets(sample_record):
    boxes = []
    labels = []
    for ann_token in sample_record['anns']:
        ann = annotation_by_token[ann_token]
        category = category_by_token[ann['category_token']]['name']
        det_class = category_to_detection_class(category)
        if det_class is None:
            continue

        x, y, z = ann['translation']
        w, l, h = ann['size']
        yaw = yaw_from_quaternion(ann['rotation'])
        boxes.append([x, y, z, w, l, h, yaw])
        labels.append(CLASS_TO_ID[det_class])

    if not boxes:
        return torch.zeros((0, 7), dtype=torch.float32), torch.zeros((0,), dtype=torch.long)

    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)


gt_boxes_ego, gt_labels = build_gt_targets(sample_record)
gt_boxes_ego.shape, gt_labels.shape

## 6. Image Preprocessing And PyTorch Dataset Contract

The job here is to turn raw JPEGs plus metadata into tensors that the model can consume directly. The end result per sample should look like:

```python
{
    'images': Tensor[N, 3, H, W],
    'img_metas': {
        'lidar2img': Tensor[N, 4, 4],
        'image_shape': Tensor[N, 2],
        'sample_token': str,
    },
    'gt_boxes_ego': Tensor[M, 7],
    'gt_labels': Tensor[M],
}
```

Shape meanings:
- `N` = number of cameras, so `N=6` for nuScenes.
- `H, W` = resized image height and width used by the model, not the raw JPEG resolution.
- `M` = number of objects in this sample. This stays variable because every scene has a different number of boxes.

The asymmetry here is intentional: images are dense and easy to stack into one tensor, while targets are sparse and variable-length, so they stay as per-sample lists until loss computation.

In [ ]:
# Minimal shape-only example before we touch the real dataset.
import torch

# B: batch size, N: number of cameras, H/W: resized image resolution used by the model.
B, N, H, W = 2, 6, 256, 448

# [B, N, 3, H, W]: two samples, six cameras each, RGB images.
images = torch.randn(B, N, 3, H, W)

# img_metas stays a Python list of length B because metadata is per-sample, not one big dense tensor.
img_metas = [
    {
        # [N, 4, 4]: one projection matrix per camera for this sample.
        'lidar2img': torch.eye(4).unsqueeze(0).repeat(N, 1, 1),
        # [N, 2]: each camera stores its resized [H, W].
        'image_shape': torch.tensor([[H, W]]).repeat(N, 1),
    }
    for _ in range(B)
]

# Targets are ragged lists because the number of objects differs by sample.
# Sample 0 has 8 boxes, sample 1 has 5 boxes.
gt_boxes_ego = [torch.randn(8, 7), torch.randn(5, 7)]
gt_labels = [torch.randint(0, 10, (8,)), torch.randint(0, 10, (5,))]

images.shape, len(img_metas), gt_boxes_ego[0].shape, gt_labels[0].shape


In [ ]:
def resize_and_normalize_image(image: Image.Image, image_size=(256, 448)):
    image = image.resize((image_size[1], image_size[0]))
    array = np.asarray(image, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    array = (array - mean) / std
    return torch.from_numpy(array).permute(2, 0, 1)


@dataclass
class NuScenesTables:
    samples: List[dict]
    sample_by_token: Dict[str, dict]
    sample_data_by_token: Dict[str, dict]
    annotation_by_token: Dict[str, dict]
    calibrated_sensor_by_token: Dict[str, dict]
    ego_pose_by_token: Dict[str, dict]
    category_by_token: Dict[str, dict]


tables = NuScenesTables(
    samples=sample_table,
    sample_by_token=sample_by_token,
    sample_data_by_token=sample_data_by_token,
    annotation_by_token=annotation_by_token,
    calibrated_sensor_by_token=calibrated_sensor_by_token,
    ego_pose_by_token=ego_pose_by_token,
    category_by_token=category_by_token,
)


class NuScenesDetr3DDataset(Dataset):
    def __init__(self, dataroot: Path, tables: NuScenesTables, image_size=(256, 448)):
        self.dataroot = dataroot
        self.tables = tables
        self.image_size = image_size

    def __len__(self):
        return len(self.tables.samples)

    def __getitem__(self, index):
        sample_record = self.tables.samples[index]
        camera_records = get_camera_records(sample_record, self.tables.sample_data_by_token)

        images = []
        lidar2img = []
        image_shape = []

        for camera_name in CAMERA_NAMES:
            record = camera_records[camera_name]
            image_path = self.dataroot / record['filename']
            image = Image.open(image_path).convert('RGB')
            image_tensor = resize_and_normalize_image(image, image_size=self.image_size)
            images.append(image_tensor)
            # One 4x4 projection matrix per camera view.
            lidar2img.append(torch.from_numpy(build_lidar2img(record)))
            # Stored as [H, W] so later projection code can normalize image coordinates.
            image_shape.append(torch.tensor(self.image_size, dtype=torch.float32))

        gt_boxes_ego, gt_labels = build_gt_targets(sample_record)

        return {
            # [N, 3, H, W] because one sample contains N synchronized camera images.
            'images': torch.stack(images, dim=0),
            'img_metas': {
                # [N, 4, 4] aligns one projection matrix with each camera image.
                'lidar2img': torch.stack(lidar2img, dim=0),
                # [N, 2] repeats the resized image size once per camera.
                'image_shape': torch.stack(image_shape, dim=0),
                'sample_token': sample_record['token'],
            },
            'gt_boxes_ego': gt_boxes_ego,
            'gt_labels': gt_labels,
        }


dataset = NuScenesDetr3DDataset(DATAROOT, tables)
sample = dataset[0]
sample['images'].shape, sample['img_metas']['lidar2img'].shape, sample['gt_boxes_ego'].shape

In [ ]:
def detr3d_collate(batch: List[dict]):
    # [B, N, 3, H, W]: batch dimension is added here, camera dimension already exists per sample.
    images = torch.stack([item['images'] for item in batch], dim=0)
    img_metas = [item['img_metas'] for item in batch]
    # GT stays ragged because each sample can contain a different number of boxes.
    gt_boxes_ego = [item['gt_boxes_ego'] for item in batch]
    gt_labels = [item['gt_labels'] for item in batch]
    return {
        'images': images,
        'img_metas': img_metas,
        'gt_boxes_ego': gt_boxes_ego,
        'gt_labels': gt_labels,
    }


batch = detr3d_collate([dataset[0], dataset[1]])
batch['images'].shape, len(batch['img_metas']), len(batch['gt_boxes_ego'])

From this point on, the model does not care about raw JSON or JPEG files anymore. It only sees the PyTorch batch contract we just built.

## 7. Backbone

The backbone extracts per-camera 2D image features. The only multi-view step here is flattening `[B, N, 3, H, W]` into `[B*N, 3, H, W]`, applying the CNN, then reshaping back to `[B, N, C, H_l, W_l]`.

Why flatten first? Because the CNN does not need to know which camera a feature came from. Each image can be processed independently with shared weights, and reshaping to `B*N` lets us reuse ordinary 2D conv layers without writing a special multi-camera backbone.

In [ ]:
class ConvNormAct(nn.Sequential):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = ConvNormAct(in_channels, out_channels, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.conv2(out)
        return self.relu(out + residual)


class MultiViewImageBackbone(nn.Module):
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.stage3 = nn.Sequential(ResidualBlock(base_channels, 128, stride=2), ResidualBlock(128, 128))
        self.stage4 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.stage5 = nn.Sequential(ResidualBlock(256, 512, stride=2), ResidualBlock(512, 512))

    def forward(self, images):
        batch, num_cams, channels, height, width = images.shape
        # Merge batch and camera dims so a standard 2D backbone can process every view identically.
        x = images.reshape(batch * num_cams, channels, height, width)
        x = self.stem(x)
        stage3 = self.stage3(x)
        stage4 = self.stage4(stage3)
        stage5 = self.stage5(stage4)

        def reshape_views(feat):
            _, c, h, w = feat.shape
            # Restore camera structure so later projection code knows which feature map belongs to which camera.
            return feat.reshape(batch, num_cams, c, h, w)

        return {'stage3': reshape_views(stage3), 'stage4': reshape_views(stage4), 'stage5': reshape_views(stage5)}


backbone = MultiViewImageBackbone()
backbone_features = backbone(batch['images'])
{name: feat.shape for name, feat in backbone_features.items()}

## 8. FPN

The neck merges multiple backbone stages into a semantic pyramid with aligned channel dimensions. DETR3D later samples from these levels when projecting 3D queries into the images.

Why do all pyramid levels end up with the same channel count? Because attention wants one common embedding size across levels. Spatial resolution can differ by level, but channel dimension should stay aligned so the sampled features can be fused cleanly.

In [ ]:
class FPNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.lateral = nn.Conv2d(in_channels, out_channels, 1)
        self.output = nn.Conv2d(out_channels, out_channels, 3, padding=1)


class ImageFPN(nn.Module):
    def __init__(self, in_channels=(128, 256, 512), out_channels=256):
        super().__init__()
        self.blocks = nn.ModuleList([FPNBlock(c, out_channels) for c in in_channels])
        self.out_names = ['p3', 'p4', 'p5']

    def forward(self, features):
        pyramid = {}
        top_down = None
        for name, block, out_name in zip(reversed(sorted(features.keys())), reversed(self.blocks), reversed(self.out_names)):
            feat = features[name]
            b, n, c, h, w = feat.shape
            # Flatten views again because FPN is also just regular 2D convs applied per image.
            flat = feat.reshape(b * n, c, h, w)
            lateral = block.lateral(flat)
            if top_down is not None:
                lateral = lateral + F.interpolate(top_down, size=lateral.shape[-2:], mode='nearest')
            out = block.output(lateral)
            top_down = lateral
            # Output shape is [B, N, C_fpn, H_l, W_l]: camera axis is preserved, only channels are unified.
            pyramid[out_name] = out.reshape(b, n, out.shape[1], out.shape[2], out.shape[3])
        return {name: pyramid[name] for name in self.out_names}


neck = ImageFPN()
pyramid = neck(backbone_features)
{name: feat.shape for name, feat in pyramid.items()}

## 9. Reference Points And Feature Sampling

DETR3D does not build dense image-space anchors. Instead, it learns object queries, maps them to normalized 3D reference points, and projects those 3D points into every camera view.

This is where most of the non-obvious shapes come from:
- `reference_points`: `[B, Q, 3]` because every query predicts one candidate 3D location `(x, y, z)`.
- `proj`: `[B, N, 4, 4]` because each sample has one projection matrix per camera.
- sampled features eventually become `[B, C, Q, N, 1, L]`: channels first for convenience, one sampled point per query (`1`), and `L` feature levels.

In [ ]:
def inverse_sigmoid(x, eps=1e-5):
    x = x.clamp(min=eps, max=1 - eps)
    return torch.log(x / (1 - x))


def denormalize_reference_points(reference_points, pc_range):
    pc_range = torch.as_tensor(pc_range, dtype=reference_points.dtype, device=reference_points.device)
    xyz_min = pc_range[:3]
    xyz_max = pc_range[3:]
    return reference_points * (xyz_max - xyz_min) + xyz_min


def feature_sampling(mlvl_feats, reference_points, pc_range, img_metas):
    batch, num_queries, _ = reference_points.shape
    _, num_cams, channels, _, _ = mlvl_feats[0].shape
    ref_points_3d = denormalize_reference_points(reference_points, pc_range)
    # Homogeneous coordinates let us apply a single 4x4 projection matrix.
    ref_homo = torch.cat([ref_points_3d, torch.ones_like(ref_points_3d[..., :1])], dim=-1)

    proj = torch.stack([meta['lidar2img'].to(reference_points.dtype) for meta in img_metas], dim=0)
    image_shapes = torch.stack([meta['image_shape'].to(reference_points.dtype) for meta in img_metas], dim=0)

    # Broadcast to [B, N, Q, 4] so every query is projected into every camera.
    proj_points = torch.matmul(proj[:, :, None], ref_homo[:, None, :, :, None]).squeeze(-1)
    depth = proj_points[..., 2:3]
    xy = proj_points[..., :2] / depth.clamp(min=1e-5)
    width = image_shapes[..., 1:2]
    height = image_shapes[..., 0:1]
    # grid_sample expects normalized image coords in [-1, 1], not pixel coords.
    grid = torch.stack([xy[..., 0] / width * 2 - 1, xy[..., 1] / height * 2 - 1], dim=-1)

    valid = (depth[..., 0] > 1e-5)
    valid = valid & (grid[..., 0] >= -1) & (grid[..., 0] <= 1) & (grid[..., 1] >= -1) & (grid[..., 1] <= 1)

    sampled_levels = []
    masks = []
    for feat in mlvl_feats:
        feat = feat.reshape(batch * num_cams, channels, feat.shape[-2], feat.shape[-1])
        # Query locations become a sampling grid of shape [B*N, Q, 1, 2].
        level_grid = grid.reshape(batch * num_cams, num_queries, 1, 2)
        sampled = F.grid_sample(feat, level_grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        # Rearrange to [B, C, Q, N, 1] so attention can later weight cameras and levels explicitly.
        sampled = sampled.reshape(batch, num_cams, channels, num_queries, 1).permute(0, 2, 3, 1, 4)
        sampled_levels.append(sampled)
        # Same validity mask is reused across levels because projection happens before level-specific sampling.
        masks.append(valid[:, None, :, :, None].to(reference_points.dtype))

    return ref_points_3d, torch.stack(sampled_levels, dim=-1), torch.stack(masks, dim=-1)


## 10. Cross-Attention

This module turns query features into weights over cameras and feature levels, samples the image pyramid at the projected 3D points, and fuses the result back into the query representation.

The important mental model is that DETR3D attention is not attending over every image pixel. It predicts a small set of weights per query over `(camera, level)` pairs, then reads features only at the projected 3D locations.

In [ ]:
class Detr3DCrossAttention(nn.Module):
    def __init__(self, embed_dims=256, num_cams=6, num_levels=3, pc_range=(-51.2, -51.2, -5.0, 51.2, 51.2, 3.0)):
        super().__init__()
        self.num_cams = num_cams
        self.num_levels = num_levels
        self.pc_range = pc_range
        self.attention_weights = nn.Linear(embed_dims, num_cams * num_levels)
        self.output_proj = nn.Linear(embed_dims, embed_dims)
        self.position_encoder = nn.Sequential(
            nn.Linear(3, embed_dims),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dims, embed_dims),
        )

    def forward(self, query, mlvl_feats, reference_points, img_metas, query_pos=None):
        attn_input = query if query_pos is None else query + query_pos
        weights = self.attention_weights(attn_input).view(query.shape[0], query.shape[1], self.num_cams, self.num_levels)
        # Reshape to [B, 1, Q, N, 1, L] so it can broadcast against sampled features [B, C, Q, N, 1, L].
        weights = weights.sigmoid().permute(0, 3, 1, 2).unsqueeze(1).unsqueeze(4)
        _, sampled_feats, mask = feature_sampling(mlvl_feats, reference_points, self.pc_range, img_metas)
        # Sum over feature levels and cameras, leaving one fused embedding per query.
        fused = (sampled_feats * weights * mask).sum(dim=-1).sum(dim=-1).squeeze(-1)
        fused = fused.permute(0, 2, 1)
        return self.output_proj(fused) + self.position_encoder(inverse_sigmoid(reference_points))


## 11. Decoder Layer And Transformer

The decoder alternates self-attention among the object queries and cross-attention from queries into the multi-view image features.

Shape convention in this section:
- query/state tensors stay `[B, Q, C]` because transformers operate on sequences of queries.
- stacking decoder outputs gives `[num_layers, B, Q, C]` so the head can supervise intermediate layers too.
- reference points stay `[B, Q, 3]` because each query keeps one current 3D anchor location.

In [ ]:
class Detr3DDecoderLayer(nn.Module):
    def __init__(self, embed_dims=256, num_heads=8, num_cams=6, num_levels=3):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dims, num_heads, batch_first=True)
        self.cross_attn = Detr3DCrossAttention(embed_dims=embed_dims, num_cams=num_cams, num_levels=num_levels)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dims, embed_dims * 4),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dims * 4, embed_dims),
        )
        self.norm1 = nn.LayerNorm(embed_dims)
        self.norm2 = nn.LayerNorm(embed_dims)
        self.norm3 = nn.LayerNorm(embed_dims)

    def forward(self, query, mlvl_feats, query_pos, reference_points, img_metas):
        attended, _ = self.self_attn(query + query_pos, query + query_pos, query)
        query = self.norm1(query + attended)
        query = self.norm2(query + self.cross_attn(query, mlvl_feats, reference_points, img_metas, query_pos=query_pos))
        return self.norm3(query + self.ffn(query))


class Detr3DTransformer(nn.Module):
    def __init__(self, embed_dims=256, num_queries=900, num_layers=6, num_heads=8, num_cams=6, num_levels=3):
        super().__init__()
        self.query_embed = nn.Embedding(num_queries, embed_dims)
        self.query_pos = nn.Embedding(num_queries, embed_dims)
        self.reference_points = nn.Linear(embed_dims, 3)
        self.layers = nn.ModuleList([
            Detr3DDecoderLayer(embed_dims=embed_dims, num_heads=num_heads, num_cams=num_cams, num_levels=num_levels)
            for _ in range(num_layers)
        ])

    def forward(self, pyramid, img_metas):
        mlvl_feats = list(pyramid.values())
        batch = mlvl_feats[0].shape[0]
        # Learned queries are shared across the batch, so we expand instead of creating different query sets per sample.
        query = self.query_embed.weight.unsqueeze(0).expand(batch, -1, -1)
        query_pos = self.query_pos.weight.unsqueeze(0).expand(batch, -1, -1)
        # Each query predicts one normalized 3D point in [0, 1]^3.
        reference_points = self.reference_points(query_pos).sigmoid()
        init_reference = reference_points
        states = []
        refs = []
        hidden = query
        for layer in self.layers:
            hidden = layer(hidden, mlvl_feats, query_pos, reference_points, img_metas)
            states.append(hidden)
            refs.append(reference_points)
        # [L, B, Q, C], [B, Q, 3], [L, B, Q, 3]
        return torch.stack(states), init_reference, torch.stack(refs)


## 12. Detection Head And Full Model

The head predicts classes and 3D boxes from each decoder layer. The full model is now just composition: backbone -> neck -> transformer -> head.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True)])
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class Detr3DHead(nn.Module):
    def __init__(self, embed_dims=256, num_classes=10, box_dim=10):
        super().__init__()
        self.cls_branch = MLP(embed_dims, embed_dims, num_classes + 1, 3)
        self.reg_branch = MLP(embed_dims, embed_dims, box_dim, 3)

    def forward(self, hs):
        cls_scores = [self.cls_branch(layer_hs) for layer_hs in hs]
        bbox_preds = [self.reg_branch(layer_hs) for layer_hs in hs]
        return torch.stack(cls_scores), torch.stack(bbox_preds)


class Detr3DModel(nn.Module):
    def __init__(self, backbone, neck, transformer, head):
        super().__init__()
        self.backbone = backbone
        self.neck = neck
        self.transformer = transformer
        self.head = head

    def forward(self, images, img_metas):
        features = self.backbone(images)
        pyramid = self.neck(features)
        hs, init_reference, inter_references = self.transformer(pyramid, img_metas)
        cls_scores, bbox_preds = self.head(hs)
        return {
            'cls_scores': cls_scores,
            'bbox_preds': bbox_preds,
            'init_reference': init_reference,
            'inter_references': inter_references,
        }


model = Detr3DModel(
    backbone=MultiViewImageBackbone(),
    neck=ImageFPN(),
    transformer=Detr3DTransformer(num_queries=100),
    head=Detr3DHead(),
)
outputs = model(batch['images'], batch['img_metas'])
# cls_scores: [num_layers, B, Q, num_classes + 1]
# bbox_preds: [num_layers, B, Q, box_dim]
# init_reference: [B, Q, 3]
# inter_references: [num_layers, B, Q, 3]
{name: value.shape for name, value in outputs.items()}

## 13. Matcher And Loss

DETR-style training matches a fixed number of predictions to a variable number of targets. Here we use a simple matching cost built from class confidence and normalized box L1 distance.

In [ ]:
def box3d_to_10d(boxes):
    out = boxes.new_zeros((boxes.shape[0], 10))
    out[:, 0] = boxes[:, 0]
    out[:, 1] = boxes[:, 1]
    out[:, 2] = boxes[:, 4]
    out[:, 3] = boxes[:, 3]
    out[:, 4] = boxes[:, 2]
    out[:, 5] = boxes[:, 5]
    out[:, 6] = torch.sin(boxes[:, 6])
    out[:, 7] = torch.cos(boxes[:, 6])
    return out


def normalize_bbox(boxes, pc_range=(-51.2, -51.2, -5.0, 51.2, 51.2, 3.0)):
    pc_range = torch.as_tensor(pc_range, dtype=boxes.dtype, device=boxes.device)
    xyz_min, xyz_max = pc_range[:3], pc_range[3:]
    xyz_span = (xyz_max - xyz_min).clamp(min=1e-5)
    out = boxes.clone()
    out[..., 0] = (boxes[..., 0] - xyz_min[0]) / xyz_span[0]
    out[..., 1] = (boxes[..., 1] - xyz_min[1]) / xyz_span[1]
    out[..., 4] = (boxes[..., 4] - xyz_min[2]) / xyz_span[2]
    out[..., 2] = boxes[..., 2] / xyz_span[0]
    out[..., 3] = boxes[..., 3] / xyz_span[1]
    out[..., 5] = boxes[..., 5] / xyz_span[2]
    return out


class HungarianMatcher3D:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    @torch.no_grad()
    def __call__(self, cls_logits, box_preds, gt_boxes, gt_labels):
        assignments = []
        for batch_idx in range(cls_logits.shape[0]):
            if gt_boxes[batch_idx].numel() == 0:
                empty = torch.empty(0, dtype=torch.long, device=cls_logits.device)
                assignments.append((empty, empty))
                continue
            gt_10d = box3d_to_10d(gt_boxes[batch_idx])
            probs = cls_logits[batch_idx].softmax(-1)[:, : self.num_classes]
            cls_cost = -probs[:, gt_labels[batch_idx]]
            bbox_cost = torch.cdist(normalize_bbox(box_preds[batch_idx])[:, :8], normalize_bbox(gt_10d)[:, :8], p=1)
            total_cost = cls_cost + bbox_cost
            taken_rows, taken_cols = [], []
            for _ in range(min(total_cost.shape)):
                flat_idx = total_cost.argmin()
                row = flat_idx // total_cost.shape[1]
                col = flat_idx % total_cost.shape[1]
                if int(row) in taken_rows or int(col) in taken_cols:
                    total_cost[row, col] = float('inf')
                    continue
                taken_rows.append(int(row))
                taken_cols.append(int(col))
                total_cost[row] = float('inf')
                total_cost[:, col] = float('inf')
            assignments.append((torch.tensor(taken_rows), torch.tensor(taken_cols)))
        return assignments


class Detr3DLoss:
    def __init__(self, num_classes, bg_cls_weight=0.1):
        self.num_classes = num_classes
        self.bg_cls_weight = bg_cls_weight
        self.matcher = HungarianMatcher3D(num_classes=num_classes)

    def loss_by_feat(self, all_cls_scores, all_bbox_preds, batch_gt_boxes7, batch_gt_labels):
        cls_scores = all_cls_scores[-1]
        bbox_preds = all_bbox_preds[-1]
        assignments = self.matcher(cls_scores, bbox_preds, batch_gt_boxes7, batch_gt_labels)

        labels = torch.full(cls_scores.shape[:2], self.num_classes, dtype=torch.long, device=cls_scores.device)
        bbox_targets = torch.zeros_like(bbox_preds)
        bbox_weights = torch.zeros_like(bbox_preds)

        num_pos = 0
        num_neg = cls_scores.shape[0] * cls_scores.shape[1]
        for batch_idx, (pred_ids, gt_ids) in enumerate(assignments):
            if pred_ids.numel() == 0:
                continue
            gt_10d = box3d_to_10d(batch_gt_boxes7[batch_idx])[gt_ids]
            labels[batch_idx, pred_ids] = batch_gt_labels[batch_idx][gt_ids]
            bbox_targets[batch_idx, pred_ids] = gt_10d
            bbox_weights[batch_idx, pred_ids] = 1.0
            num_pos += pred_ids.numel()
            num_neg -= pred_ids.numel()

        cls_avg_factor = max(float(num_pos) + self.bg_cls_weight * float(num_neg), 1.0)
        loss_cls = F.cross_entropy(cls_scores.reshape(-1, cls_scores.shape[-1]), labels.reshape(-1), reduction='sum') / cls_avg_factor
        loss_bbox = (F.l1_loss(normalize_bbox(bbox_preds), normalize_bbox(bbox_targets), reduction='none') * bbox_weights).sum() / max(float(num_pos), 1.0)
        return {'loss_cls': loss_cls, 'loss_bbox': loss_bbox}


criterion = Detr3DLoss(num_classes=10)
loss_dict = criterion.loss_by_feat(outputs['cls_scores'], outputs['bbox_preds'], batch['gt_boxes_ego'], batch['gt_labels'])
loss_dict

## 14. Training Step And Experiments

Once the batch contract, model, and loss are explicit, the training loop becomes straightforward. The important debugging phases are:
1. one sample
2. one batch
3. tiny split
4. full trainval

In [ ]:
def train_one_epoch(model, criterion, dataloader, optimizer, device):
    model.train()
    running = 0.0
    for batch in dataloader:
        images = batch['images'].to(device)
        img_metas = [
            {
                'lidar2img': meta['lidar2img'].to(device),
                'image_shape': meta['image_shape'].to(device),
                'sample_token': meta['sample_token'],
            }
            for meta in batch['img_metas']
        ]
        gt_boxes = [boxes.to(device) for boxes in batch['gt_boxes_ego']]
        gt_labels = [labels.to(device) for labels in batch['gt_labels']]

        outputs = model(images, img_metas)
        loss_dict = criterion.loss_by_feat(outputs['cls_scores'], outputs['bbox_preds'], gt_boxes, gt_labels)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += float(loss.detach().cpu())
    return {'loss': running / max(len(dataloader), 1)}


# Suggested next experiments:
# 1. overfit dataset[0]
# 2. overfit 2-4 samples
# 3. log cls/bbox loss separately
# 4. visualize projected GT centers and predicted queries